# Data, tensor, context, and expert parallelism

Implement nine parallelism strategies with CPU `torch.Tensor` values on $P$ `flock` ranks.

## Conventions

- Every dimension sharded by a puzzle is divisible by $P$.
- Shards are contiguous and assigned in rank order.
- $B$ is the batch or token count, $D$ is the model dimension, $F$ is the MLP hidden dimension, $S$ is the sequence length, $H$ is the number of attention heads, $D_h = D/H$, and $V$ is the vocabulary size.
- Each rank may access only the tensor and parameter shards assigned to it.
- Communication must use the tensor primitives from the preceding puzzles; PyTorch distributed APIs are not allowed.
- Correctness is evaluated against an equivalent single-rank PyTorch computation. The DDP puzzle includes backward and one optimizer step; the remaining puzzles evaluate forward values only.

In [1]:
import flock
import flock.torch as ft
import torch

## 1. Distributed data parallel

Replicate the same model and initial parameters on every rank. Rank $r$ receives a distinct local minibatch $(X_r, \operatorname{target}_r)$ of size $B/P$.

Perform one synchronous training step:

1. Run the local forward pass and compute the local mean loss.
2. Use autograd to compute local parameter gradients.
3. Sum each gradient across all ranks and divide by $P$.
4. Apply one identical SGD update on every rank.

Do not exchange input samples, activations, or parameters. After the update, all parameter replicas must be equal. They must also match a single-rank reference step computed from the concatenation of all local minibatches with a mean loss over the global batch.

In [2]:
world_size = 4
local_batch_size = 10
learning_rate = 0.1


def make_model():
    torch.manual_seed(42)
    return torch.nn.Sequential(
        torch.nn.Linear(10, 10),
        torch.nn.ReLU(),
        torch.nn.Linear(10, 1),
    )


def make_batch(rank: int):
    generator = torch.Generator().manual_seed(1_000 + rank)
    X = torch.rand((local_batch_size, 10), generator=generator)
    y = (torch.rand((local_batch_size, 1), generator=generator) < 0.5).float()
    return X, y


@flock.distribute(world_size, policy=flock.Fifo())
async def ddp():
    model = make_model()
    optim = torch.optim.SGD(model.parameters(), lr=learning_rate)
    X, y = make_batch(flock.get_rank())

    loss = torch.binary_cross_entropy_with_logits(model(X), y).mean()
    loss.backward()

    for param in model.parameters():
        grad = param.grad
        assert grad is not None
        reduced = await ft.all_reduce(grad, "sum").wait()
        grad.copy_(reduced / flock.get_world_size())

    optim.step()
    return tuple(param.detach().clone() for param in model.parameters())


result = ddp()

reference = make_model()
reference_optim = torch.optim.SGD(reference.parameters(), lr=learning_rate)
batches = [make_batch(rank) for rank in range(world_size)]
X = torch.cat([batch[0] for batch in batches])
y = torch.cat([batch[1] for batch in batches])
reference_loss = torch.binary_cross_entropy_with_logits(reference(X), y).mean()
reference_loss.backward()
reference_optim.step()
expected = tuple(param.detach() for param in reference.parameters())

for rank_params in result:
    for actual, want in zip(rank_params, expected, strict=True):
        torch.testing.assert_close(actual, want)

## 2. Column-wise tensor-parallel linear

Let

$$
X \in \mathbb{R}^{B \times D_{\mathrm{in}}}, \qquad
W \in \mathbb{R}^{D_{\mathrm{in}} \times D_{\mathrm{out}}}, \qquad
b \in \mathbb{R}^{D_{\mathrm{out}}}.
$$

The reference operation is

$$
Y = XW + b.
$$

$X$ is replicated. Rank $r$ owns

$$
W_r = W\!\left[:,\frac{rD_{\mathrm{out}}}{P}:\frac{(r+1)D_{\mathrm{out}}}{P}\right]
$$

and the corresponding bias shard $b_r$. Compute the local output shard

$$
Y_r = XW_r + b_r.
$$

All-gather the local shards along the last dimension and return the replicated full output

$$
Y = \operatorname{concat}(Y_0, \ldots, Y_{P-1}) \in \mathbb{R}^{B \times D_{\mathrm{out}}}.
$$

In [3]:
world_size = 100


@flock.distribute(world_size)
async def columnwise_tp(X: torch.Tensor, W: torch.Tensor, b: torch.Tensor):
    part = X @ W + b[None, :]

    full = torch.cat(await ft.all_gather(part).wait(), dim=-1)

    return full


X = torch.rand(10, 16)
W = torch.rand(16, 10 * world_size)
b = torch.rand((10 * world_size,))
expected = X @ W + b[None, :]

result = columnwise_tp(
    X,
    flock.per_rank(W.chunk(world_size, dim=1)),
    flock.per_rank(b.chunk(world_size, dim=0)),
)

assert all(torch.allclose(expected, f) for f in result)

## 3. Row-wise tensor-parallel linear

Let

$$
X \in \mathbb{R}^{B \times D_{\mathrm{in}}}, \qquad
W \in \mathbb{R}^{D_{\mathrm{in}} \times D_{\mathrm{out}}}, \qquad
b \in \mathbb{R}^{D_{\mathrm{out}}},
$$

with reference output

$$
Y = XW + b.
$$

Rank $r$ owns matching input and weight shards

$$
\begin{aligned}
X_r &= X\!\left[:,\frac{rD_{\mathrm{in}}}{P}:\frac{(r+1)D_{\mathrm{in}}}{P}\right], \\
W_r &= W\!\left[\frac{rD_{\mathrm{in}}}{P}:\frac{(r+1)D_{\mathrm{in}}}{P},:\right].
\end{aligned}
$$

Compute the local partial result

$$
Z_r = X_r W_r,
$$

then sum all partial results across the tensor-parallel group. Apply $b$ exactly once after the sum.

Return a replicated tensor $Y$ with shape $[B, D_{\mathrm{out}}]$ on every rank. Each result must equal the complete reference output.

In [4]:
world_size = 100


@flock.distribute(world_size)
async def rowwise_tp(X: torch.Tensor, W: torch.Tensor, b: torch.Tensor):
    part = X @ W
    full = await ft.all_reduce(part, "sum").wait()
    return full + b[None, :]


X = torch.rand(16, 4 * world_size)
W = torch.rand(4 * world_size, 30)
b = torch.rand((30,))
expected = X @ W + b[None, :]


result = rowwise_tp(
    flock.per_rank(X.chunk(world_size, dim=1)),
    flock.per_rank(W.chunk(world_size, dim=0)),
    b,
)

assert all(torch.allclose(expected, f) for f in result)

## 4. Tensor-parallel MLP

Implement the reference MLP

$$
Y = \operatorname{gelu}(XW_0 + b_0)W_1 + b_1,
$$

where

$$
X \in \mathbb{R}^{B \times D}, \qquad
W_0 \in \mathbb{R}^{D \times F}, \qquad
W_1 \in \mathbb{R}^{F \times D}.
$$

Use column-wise parallelism for the first linear layer and row-wise parallelism for the second:

- $X$ and $Y$ are replicated.
- Rank $r$ owns columns $rF/P:(r+1)F/P$ of $W_0$ and the corresponding shard of $b_0$.
- Rank $r$ owns rows $rF/P:(r+1)F/P$ of $W_1$.
- The GELU activation remains sharded with shape $[B, F/P]$ and must never be gathered.
- $b_1$ is applied exactly once after combining the partial outputs of the second linear layer.

Return the complete reference output with shape $[B, D]$ on every rank.

In [5]:
world_size = 100


@flock.distribute(world_size)
async def col_row_mlp(
    X: torch.Tensor,
    W0: torch.Tensor,
    W1: torch.Tensor,
    b0: torch.Tensor,
    b1: torch.Tensor,
):
    hidden = torch.nn.functional.gelu(X @ W0 + b0[None, :])
    part = hidden @ W1
    full = await ft.all_reduce(part).wait()
    return full + b1[None, :]


X = torch.rand(16, 32)
W0, b0 = torch.rand(32, 10 * world_size), torch.rand((10 * world_size,))
W1, b1 = torch.rand(10 * world_size, 32), torch.rand((32,))

expected = torch.nn.functional.gelu(X @ W0 + b0[None, :]) @ W1 + b1[None, :]

result = col_row_mlp(
    X,
    flock.per_rank(W0.chunk(world_size, dim=1)),
    flock.per_rank(W1.chunk(world_size, dim=0)),
    flock.per_rank(b0.chunk(world_size, dim=0)),
    b1,
)

assert all(torch.allclose(expected, x) for x in result)


## 5. Ulysses context-parallel attention

Inputs $Q_r$, $K_r$, and $V_r$ are initially sequence-sharded. Rank $r$ owns

$$
Q_r, K_r, V_r \in \mathbb{R}^{B \times (S/P) \times H \times D_h},
$$

corresponding to sequence positions $rS/P:(r+1)S/P$ and all attention heads.

Implement the following layout transformation for each of $Q$, $K$, and $V$:

1. Split the local tensor into $P$ contiguous head shards.
2. Send head shard $j$ to rank $j$ with an all-to-all exchange.
3. Concatenate received tensors along the sequence dimension.

After the exchange, rank $r$ must own the complete sequence for heads $rH/P:(r+1)H/P$, with layout

$$
\mathbb{R}^{B \times S \times (H/P) \times D_h}.
$$

Compute non-causal scaled dot-product attention locally, without dropout. Apply the inverse all-to-all transformation: split along the sequence dimension, exchange the pieces, and concatenate along the head dimension.

Return a sequence-sharded tensor in $\mathbb{R}^{B \times (S/P) \times H \times D_h}$. Rank $r$ must match sequence shard $r$ of the single-rank attention reference.

In [6]:
world_size = 16


@flock.distribute(world_size)
async def ulysses(X: torch.Tensor, W_qkv: torch.Tensor, num_heads: int):
    ws = flock.get_world_size()
    B, s, E = X.shape
    assert E % num_heads == 0
    head_dim = E // num_heads

    qkv = X @ W_qkv
    qkv = qkv.reshape(B, s, 3, num_heads, head_dim)

    qkv_full_seq = await ft.all_to_all(qkv.chunk(ws, dim=-2)).wait()

    qkv = torch.cat(qkv_full_seq, dim=1)

    assert qkv.shape == (B, s * ws, 3, num_heads // ws, head_dim)

    q, k, v = qkv.unbind(dim=2)

    q = q.transpose(1, 2)
    k = k.transpose(1, 2)
    v = v.transpose(1, 2)
    output = torch.softmax(q @ k.mT / head_dim**0.5, dim=-1) @ v
    output = output.transpose(1, 2)

    assert output.shape == (B, s * ws, num_heads // ws, head_dim)

    output_full_heads = await ft.all_to_all(output.chunk(ws, dim=1)).wait()
    output = torch.cat(output_full_heads, dim=-2)

    assert output.shape == (B, s, num_heads, head_dim), output.shape

    return output


embedding_dim = 128
num_heads = 16
head_dim = embedding_dim // num_heads
B = 2
S = 10 * world_size

X = torch.rand(B, S, embedding_dim)
W_qkv = torch.rand(128, 3 * embedding_dim)

qkv = X @ W_qkv

qkv = qkv.reshape(B, S, 3, num_heads, head_dim)
q, k, v = qkv.unbind(dim=2)

q = q.transpose(1, 2)
k = k.transpose(1, 2)
v = v.transpose(1, 2)


expected = torch.softmax(q @ k.mT / head_dim**0.5, dim=-1) @ v
expected = expected.transpose(1, 2)

result = ulysses(
    flock.per_rank(X.chunk(world_size, 1)),
    W_qkv,
    num_heads,
)

torch.testing.assert_close(expected, torch.cat(result, dim=1))

## 6. Ring Attention

### Objective

Compute exact non-causal scaled dot-product attention while keeping the sequence dimension sharded throughout the operation.

Rank $r$ starts with contiguous sequence shards

$$
Q_r, K_r, V_r \in \mathbb{R}^{B \times (S/P) \times H \times D_h}.
$$

$Q_r$ remains on rank $r$. K/V blocks circulate around a ring: rank $r$ sends its current block to $(r+1) \bmod P$ and receives the next block from $(r-1) \bmod P$. Starting with its local block, every rank processes exactly $P$ K/V blocks and therefore attends its local queries to all $S$ key/value positions. The complete K or V tensor must never be materialized on any rank.

### Why block outputs cannot be averaged

Attention applies one softmax across every key position. A softmax computed independently for each K/V block has a different normalization denominator, so separately normalized block outputs cannot be summed or averaged. Maintain an online softmax state that combines each block before final normalization.

For each local query and head, maintain:

- $m$: largest score seen so far, initialized to $-\infty$;
- $\ell$: sum of exponentials relative to $m$, initialized to $0$;
- $O$: unnormalized weighted-value accumulator, initialized to $0$.

For the current K/V block, compute scaled scores

$$
A = \frac{Q_r K_{\mathrm{block}}^{\mathsf T}}{\sqrt{D_h}}.
$$

Reduce only across the key positions in this block:

$$
\begin{aligned}
m_{\mathrm{block}} &= \max(A), \\
P_{\mathrm{block}} &= \exp\!\left(A - m_{\mathrm{block}}\right), \\
\ell_{\mathrm{block}} &= \sum P_{\mathrm{block}}, \\
O_{\mathrm{block}} &= P_{\mathrm{block}} V_{\mathrm{block}}.
\end{aligned}
$$

Merge the block state with the running state:

$$
\begin{aligned}
m_{\mathrm{new}} &= \max\!\left(m, m_{\mathrm{block}}\right), \\
\alpha &= \exp\!\left(m - m_{\mathrm{new}}\right), \\
\beta &= \exp\!\left(m_{\mathrm{block}} - m_{\mathrm{new}}\right), \\
\ell_{\mathrm{new}} &= \alpha \ell + \beta \ell_{\mathrm{block}}, \\
O_{\mathrm{new}} &= \alpha O + \beta O_{\mathrm{block}}.
\end{aligned}
$$

Replace $(m, \ell, O)$ with $(m_{\mathrm{new}}, \ell_{\mathrm{new}}, O_{\mathrm{new}})$ after every block. After all $P$ blocks, return

$$
\frac{O}{\ell}.
$$

### Communication requirements

- Use a ring with $\mathrm{left} = (r-1) \bmod P$ and $\mathrm{right} = (r+1) \bmod P$.
- Compute once with the local K/V block and receive the remaining $P-1$ blocks from the left neighbor.
- Forward each received block to the right neighbor until every rank has processed every block.
- Keep at most one current and one in-flight K/V block per rank; do not all-gather K or V.
- Use nonblocking communication so posting the next transfer does not require completing the current attention computation first.
- The number of heads does not need to be divisible by $P$, because heads are not sharded.

Return a local output in $\mathbb{R}^{B \times (S/P) \times H \times D_h}$. Rank $r$ must match sequence shard $r$ of a single-rank non-causal scaled dot-product attention reference, within floating-point tolerance.

In [12]:
world_size = 32


@flock.distribute(world_size, trace=True)
async def ring(X: torch.Tensor, W_qkv: torch.Tensor, num_heads: int):
    ws = flock.get_world_size()

    B, s, E = X.shape
    S = s * ws
    del s  # do not use s

    assert E % num_heads == 0
    head_dim = E // num_heads

    qkv = X @ W_qkv
    qkv = qkv.reshape(B, S // ws, 3, num_heads, head_dim)

    q, k, v = qkv.unbind(dim=2)

    q = q.transpose(1, 2)
    local_k = k.transpose(1, 2)
    local_v = v.transpose(1, 2)

    run_max = torch.full((B, num_heads, S // ws, 1), fill_value=-torch.inf)
    run_lse = torch.zeros((B, num_heads, S // ws, 1))
    run_O = torch.zeros((B, num_heads, S // ws, head_dim))

    send_works: list[flock.Work[None]] = []
    r = flock.get_rank()
    for i in range(ws):
        if i != 0:
            recv_work = ft.irecv((r - 1) % ws)
            send_work = ft.isend((r + 1) % ws, torch.cat((k.unsqueeze(0), v.unsqueeze(0)), dim=0))
            send_works.append(send_work)

            k, v = (await recv_work.wait()).unbind(dim=0)
        else:
            k = local_k
            v = local_v

        A = q @ k.mT / head_dim**0.5  # (B, num_heads, S // ws, S // ws)

        block_max = A.max(dim=-1, keepdim=True).values

        P = torch.exp(A - block_max)
        block_lse = P.sum(dim=-1, keepdim=True)

        O_block = P @ v

        old_max = run_max
        run_max = torch.maximum(run_max, block_max)

        alpha = torch.exp(old_max - run_max)
        beta = torch.exp(block_max - run_max)

        run_lse = alpha * run_lse + beta * block_lse
        run_O = alpha * run_O + beta * O_block

    run_O /= run_lse
    run_O = run_O.transpose(1, 2)

    [await w.wait() for w in send_works]

    return run_O


embedding_dim = 16
num_heads = 4
head_dim = embedding_dim // num_heads
B = 2
S = 2 * world_size

X = torch.rand(B, S, embedding_dim)
W_qkv = torch.rand(embedding_dim, 3 * embedding_dim)

qkv = X @ W_qkv

qkv = qkv.reshape(B, S, 3, num_heads, head_dim)
q, k, v = qkv.unbind(dim=2)

q = q.transpose(1, 2)
k = k.transpose(1, 2)
v = v.transpose(1, 2)


expected = torch.softmax(q @ k.mT / head_dim**0.5, dim=-1) @ v
expected = expected.transpose(1, 2)

result, trace = ring(
    flock.per_rank(X.chunk(world_size, dim=1)),
    W_qkv,
    num_heads,
)

torch.testing.assert_close(expected, torch.cat(result, dim=1))

In [8]:
print(trace.format())

communication trace
───────────────────

p2p  ·  32 ranks
  messages  992
  bytes     507904

per rank
rank  sent B  recv B  sent msg  recv msg
────────────────────────────────────────
   0   15872   15872        31        31
   1   15872   15872        31        31
   2   15872   15872        31        31
   3   15872   15872        31        31
   4   15872   15872        31        31
   5   15872   15872        31        31
   6   15872   15872        31        31
   7   15872   15872        31        31
   8   15872   15872        31        31
   9   15872   15872        31        31
  10   15872   15872        31        31
  11   15872   15872        31        31
  12   15872   15872        31        31
  13   15872   15872        31        31
  14   15872   15872        31        31
  15   15872   15872        31        31
  16   15872   15872        31        31
  17   15872   15872        31        31
  18   15872   15872        31        31
  19   15872   15872        31      

## 7. Expert parallelism

Let $E$ be the number of experts, with $E$ divisible by $P$. Expert IDs are assigned contiguously: rank $r$ owns experts

$$
\frac{rE}{P}:\frac{(r+1)E}{P}
$$

and no others.

Rank $r$ starts with a local token tensor

$$
X_r \in \mathbb{R}^{T_r \times D}
$$

and a provided top-1 expert ID for each token. Implement the following dispatch and combine procedure:

1. Group local tokens by the rank that owns their selected expert.
2. Send each group and its expert IDs to the owning rank. The all-to-all source slot identifies the source rank; retain original token indices locally.
3. Evaluate every received token with the selected local expert.
4. Return expert outputs to their source ranks.
5. Restore the original local token order.

A rank may send or receive zero tokens for any peer. Expert parameters must remain local and must not be replicated. Ignore router computation, routing weights, capacity limits, token dropping, and load-balancing losses.

Return

$$
Y_r \in \mathbb{R}^{T_r \times D}.
$$

It must match the local token shard produced by a single-rank reference using the same experts and assignments.

In [ ]:
world_size = 4


@flock.distribute(world_size, trace=True)
async def expert_parallel(
    X: torch.Tensor,
    expert_ids: torch.Tensor,
    expert_weights: torch.Tensor,
    expert_biases: torch.Tensor,
):
    rank = flock.get_rank()
    ws = flock.get_world_size()
    experts_per_rank = expert_weights.shape[0]
    owners = expert_ids // experts_per_rank
    send_indices = [torch.where(owners == dst)[0] for dst in range(ws)]
    received_tokens = await ft.all_to_all([X[indices] for indices in send_indices]).wait()
    received_ids = await ft.all_to_all([expert_ids[indices] for indices in send_indices]).wait()

    expert_start = rank * experts_per_rank
    output_sends = []
    for tokens, ids in zip(received_tokens, received_ids, strict=True):
        local_ids = ids - expert_start
        output = torch.einsum("td,tde->te", tokens, expert_weights[local_ids])
        output_sends.append(torch.nn.functional.gelu(output + expert_biases[local_ids]))

    received_outputs = await ft.all_to_all(output_sends).wait()

    Y = torch.empty_like(X)
    for indices, outputs in zip(send_indices, received_outputs, strict=True):
        Y[indices] = outputs
    return Y


expert_count = 2 * world_size
experts_per_rank = expert_count // world_size
model_dim = 8
generator = torch.Generator().manual_seed(7)
expert_weights = torch.randn(expert_count, model_dim, model_dim, generator=generator)
expert_biases = torch.randn(expert_count, model_dim, generator=generator)
token_counts = [0, 5, 2, 7]
inputs = [torch.randn(count, model_dim, generator=generator) for count in token_counts]
assignments = [
    torch.randint(expert_count, (count,), generator=generator)
    for count in token_counts
]

result, expert_trace = expert_parallel(
    flock.per_rank(inputs),
    flock.per_rank(assignments),
    flock.per_rank(expert_weights.chunk(world_size)),
    flock.per_rank(expert_biases.chunk(world_size)),
)

for X, expert_ids, actual in zip(inputs, assignments, result, strict=True):
    expected = torch.einsum("td,tde->te", X, expert_weights[expert_ids])
    expected += expert_biases[expert_ids]
    torch.testing.assert_close(actual, torch.nn.functional.gelu(expected))

assert expert_trace.summary().collective_counts == (("all_to_all", 3),)

## 8. Tensor-parallel self-attention

Implement non-causal multi-head self-attention using one-dimensional tensor parallelism. Let

$$
X \in \mathbb{R}^{B \times S \times D},
\qquad D = H D_h,
$$

and assume $H$ is divisible by $P$. $X$ is replicated. Rank $r$ owns the output-feature shards

$$
W_{Q,r}, W_{K,r}, W_{V,r} \in \mathbb{R}^{D \times (D/P)}
$$

and matching bias shards. These projections produce only heads

$$
rH/P:(r+1)H/P.
$$

Reshape each local projection to

$$
Q_r,K_r,V_r \in \mathbb{R}^{B \times (H/P) \times S \times D_h}
$$

and compute attention locally:

$$
A_r = \operatorname{softmax}\!\left(\frac{Q_rK_r^\top}{\sqrt{D_h}}\right)V_r.
$$

No communication is required between attention heads. Flatten the local heads into

$$
A_r \in \mathbb{R}^{B \times S \times (D/P)}.
$$

The output projection is row-parallel. Rank $r$ owns

$$
W_{O,r} \in \mathbb{R}^{(D/P) \times D}
$$

and computes a partial output

$$
Y_r = A_rW_{O,r}.
$$

Sum partial outputs with one all-reduce, then add the replicated output bias exactly once:

$$
Y = \sum_{r=0}^{P-1}Y_r + b_O.
$$

Return replicated $Y \in \mathbb{R}^{B \times S \times D}$. It must match a single-rank attention reference assembled from the same parameter shards. Do not all-gather Q, K, V, attention matrices, or local heads. The output all-reduce should be the only communication.

In [ ]:
world_size = 4


@flock.distribute(world_size, trace=True)
async def tensor_parallel_attention(
    X: torch.Tensor,
    W_q: torch.Tensor,
    W_k: torch.Tensor,
    W_v: torch.Tensor,
    b_q: torch.Tensor,
    b_k: torch.Tensor,
    b_v: torch.Tensor,
    W_o: torch.Tensor,
    b_o: torch.Tensor,
    num_heads: int,
):
    B, S, D = X.shape
    ws = flock.get_world_size()
    head_dim = D // num_heads
    local_heads = num_heads // ws

    def project(weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
        return (X @ weight + bias).reshape(B, S, local_heads, head_dim).transpose(1, 2)

    q = project(W_q, b_q)
    k = project(W_k, b_k)
    v = project(W_v, b_v)
    attention = torch.softmax(q @ k.mT / head_dim**0.5, dim=-1) @ v
    partial = attention.transpose(1, 2).reshape(B, S, D // ws) @ W_o
    output = await ft.all_reduce(partial, "sum").wait()
    return output + b_o


B, S, D, H = 2, 5, 32, 8
generator = torch.Generator().manual_seed(8)
X = torch.randn(B, S, D, generator=generator)
W_q = torch.randn(D, D, generator=generator)
W_k = torch.randn(D, D, generator=generator)
W_v = torch.randn(D, D, generator=generator)
b_q = torch.randn(D, generator=generator)
b_k = torch.randn(D, generator=generator)
b_v = torch.randn(D, generator=generator)
W_o = torch.randn(D, D, generator=generator)
b_o = torch.randn(D, generator=generator)

result, attention_trace = tensor_parallel_attention(
    X,
    flock.per_rank(W_q.chunk(world_size, dim=1)),
    flock.per_rank(W_k.chunk(world_size, dim=1)),
    flock.per_rank(W_v.chunk(world_size, dim=1)),
    flock.per_rank(b_q.chunk(world_size)),
    flock.per_rank(b_k.chunk(world_size)),
    flock.per_rank(b_v.chunk(world_size)),
    flock.per_rank(W_o.chunk(world_size, dim=0)),
    b_o,
    H,
)

head_dim = D // H
q = (X @ W_q + b_q).reshape(B, S, H, head_dim).transpose(1, 2)
k = (X @ W_k + b_k).reshape(B, S, H, head_dim).transpose(1, 2)
v = (X @ W_v + b_v).reshape(B, S, H, head_dim).transpose(1, 2)
attention = (torch.softmax(q @ k.mT / head_dim**0.5, dim=-1) @ v).transpose(1, 2)
expected = attention.reshape(B, S, D) @ W_o + b_o

for actual in result:
    torch.testing.assert_close(actual, expected, rtol=1e-4, atol=1e-4)

assert attention_trace.summary().collective_counts == (("all_reduce", 1),)

## 9. Vocabulary-parallel embedding and cross-entropy

Let the vocabulary size $V$ be divisible by $P$. Rank $r$ owns vocabulary IDs

$$
\mathcal{V}_r = \{rV/P,\ldots,(r+1)V/P-1\}
$$

and only its rows of an embedding matrix

$$
E_r \in \mathbb{R}^{(V/P) \times D}.
$$

### Vocabulary-parallel embedding

Every rank receives the same token IDs $T \in \{0,\ldots,V-1\}^{B \times S}$. Rank $r$ looks up IDs in $\mathcal{V}_r$ using local indices and emits zero vectors for all other IDs. Sum these local contributions with all-reduce:

$$
X = \sum_{r=0}^{P-1} X_r.
$$

Return replicated $X \in \mathbb{R}^{B \times S \times D}$ matching a full embedding lookup. No rank may materialize the full embedding matrix.

### Vocabulary-parallel cross-entropy

Given replicated hidden states $X \in \mathbb{R}^{B \times D}$, rank $r$ computes only its vocabulary shard of the logits:

$$
L_r = XE_r^\top + b_r
\in \mathbb{R}^{B \times (V/P)}.
$$

For numerical stability, compute the global maximum for each example:

$$
m_b = \max_r\max_{j \in \mathcal{V}_r} L_{r,bj},
$$

then the global softmax denominator:

$$
z_b = \sum_r\sum_{j \in \mathcal{V}_r}\exp(L_{r,bj}-m_b).
$$

For target ID $t_b$, only its owner contributes the target logit; every other rank contributes zero. Recover it with a sum all-reduce:

$$
q_b = \sum_r \mathbf{1}[t_b \in \mathcal{V}_r]L_{r,b,t_b-rV/P}.
$$

Return the mean loss

$$
\mathcal{L} = \frac{1}{B}\sum_{b=1}^{B}\left(\log z_b + m_b - q_b\right).
$$

The embedding output and loss must match full single-rank PyTorch references. Do not all-gather weights or logits. Use a max all-reduce and sum all-reduces only over vectors of size $B$ (or $B \times S$ for token-wise loss).

In [ ]:
world_size = 4


@flock.distribute(world_size, trace=True)
async def vocab_parallel_embedding(
    token_ids: torch.Tensor,
    weight: torch.Tensor,
):
    rank = flock.get_rank()
    shard_size = weight.shape[0]
    start = rank * shard_size
    owned = (token_ids >= start) & (token_ids < start + shard_size)
    local_ids = torch.where(owned, token_ids - start, 0)
    local = torch.nn.functional.embedding(local_ids, weight) * owned.unsqueeze(-1)
    return await ft.all_reduce(local, "sum").wait()


@flock.distribute(world_size, trace=True)
async def vocab_parallel_cross_entropy(
    X: torch.Tensor,
    target: torch.Tensor,
    weight: torch.Tensor,
    bias: torch.Tensor,
):
    rank = flock.get_rank()
    shard_size = weight.shape[0]
    start = rank * shard_size
    logits = X @ weight.mT + bias

    local_max = logits.max(dim=-1).values
    global_max = await ft.all_reduce(local_max, torch.maximum).wait()

    local_sum = torch.exp(logits - global_max.unsqueeze(-1)).sum(dim=-1)
    denominator = await ft.all_reduce(local_sum, "sum").wait()

    owned = (target >= start) & (target < start + shard_size)
    local_target = torch.where(owned, target - start, 0)
    target_logits = logits.gather(1, local_target.unsqueeze(1)).squeeze(1)
    target_logits = torch.where(owned, target_logits, 0.0)
    target_logits = await ft.all_reduce(target_logits, "sum").wait()

    return (denominator.log() + global_max - target_logits).mean()


vocab_size = 16 * world_size
embedding_dim = 12
batch_size = 6
sequence_length = 5
generator = torch.Generator().manual_seed(9)
embedding_weight = torch.randn(vocab_size, embedding_dim, generator=generator)
embedding_bias = torch.randn(vocab_size, generator=generator)
token_ids = torch.randint(vocab_size, (batch_size, sequence_length), generator=generator)
hidden = torch.randn(batch_size, embedding_dim, generator=generator)
target = torch.randint(vocab_size, (batch_size,), generator=generator)
weight_shards = embedding_weight.chunk(world_size)
bias_shards = embedding_bias.chunk(world_size)

embedding_result, embedding_trace = vocab_parallel_embedding(
    token_ids,
    flock.per_rank(weight_shards),
)
expected_embedding = torch.nn.functional.embedding(token_ids, embedding_weight)
for actual in embedding_result:
    torch.testing.assert_close(actual, expected_embedding)

loss_result, loss_trace = vocab_parallel_cross_entropy(
    hidden,
    target,
    flock.per_rank(weight_shards),
    flock.per_rank(bias_shards),
)
expected_loss = torch.nn.functional.cross_entropy(
    hidden @ embedding_weight.mT + embedding_bias, target
)
for actual in loss_result:
    torch.testing.assert_close(actual, expected_loss)

assert embedding_trace.summary().collective_counts == (("all_reduce", 1),)
assert loss_trace.summary().collective_counts == (("all_reduce", 3),)